# Notebook 00: DVC & MLflow Setup

**Purpose:** Initialize version control for data (DVC) and experiment tracking (MLflow)

**Key Tasks:**
1. Initialize DVC and configure remote storage
2. Add raw audio data to DVC tracking
3. Set up MLflow experiment tracking
4. Load and validate `params.yaml` configuration
5. Verify directory structure and environment

---

## 1. Import Libraries and Setup

In [1]:
import os
import sys
import yaml
import subprocess
from pathlib import Path
import json
from datetime import datetime

# MLflow for experiment tracking
import mlflow
import mlflow.sklearn

print("✓ Libraries imported successfully")
print(f"Python version: {sys.version}")
print(f"Current working directory: {os.getcwd()}")

✓ Libraries imported successfully
Python version: 3.9.6 (default, Aug  8 2025, 19:06:38) 
[Clang 17.0.0 (clang-1700.3.19.1)]
Current working directory: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/notebooks


## 2. Define Project Paths

In [2]:
# Set project root (BikeAIv5 directory)
PROJECT_ROOT = Path("/Users/harryirving/Development/projects/ai-ml/BikeAIv5")
os.chdir(PROJECT_ROOT)

# Define key directories
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
FEATURES_DIR = DATA_DIR / "features"
MODELS_DIR = PROJECT_ROOT / "models"
LOGS_DIR = PROJECT_ROOT / "logs"
RESULTS_DIR = PROJECT_ROOT / "results"
CONFIG_DIR = PROJECT_ROOT / "config"

print("Project Structure:")
print(f"  Project Root: {PROJECT_ROOT}")
print(f"  Data Dir: {DATA_DIR}")
print(f"  Raw Data: {RAW_DATA_DIR}")
print(f"  Models: {MODELS_DIR}")
print(f"  Logs: {LOGS_DIR}")

Project Structure:
  Project Root: /Users/harryirving/Development/projects/ai-ml/BikeAIv5
  Data Dir: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data
  Raw Data: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/raw
  Models: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/models
  Logs: /Users/harryirving/Development/projects/ai-ml/BikeAIv5/logs


## 3. Verify Directory Structure

Check that all required directories exist and create any missing ones.

In [3]:
# Required directory structure
required_dirs = [
    RAW_DATA_DIR / "grinder",
    RAW_DATA_DIR / "background",
    RAW_DATA_DIR / "tools",
    PROCESSED_DATA_DIR,
    FEATURES_DIR / "classical",
    FEATURES_DIR / "neural",
    MODELS_DIR,
    LOGS_DIR,
    RESULTS_DIR / "figures",
    CONFIG_DIR
]

print("Verifying directory structure...\n")
for directory in required_dirs:
    if directory.exists():
        print(f"✓ {directory.relative_to(PROJECT_ROOT)}")
    else:
        directory.mkdir(parents=True, exist_ok=True)
        print(f"✓ Created: {directory.relative_to(PROJECT_ROOT)}")

print("\n✓ All directories verified")

Verifying directory structure...

✓ data/raw/grinder
✓ data/raw/background
✓ data/raw/tools
✓ data/processed
✓ data/features/classical
✓ data/features/neural
✓ models
✓ logs
✓ results/figures
✓ config

✓ All directories verified


## 4. Check Raw Data Status

Verify that raw audio files are organized correctly.

In [4]:
def count_audio_files(directory):
    """Count audio files in a directory."""
    audio_extensions = ['.wav', '.mp3', '.flac', '.ogg', '.m4a']
    if not directory.exists():
        return 0
    return sum(1 for f in directory.iterdir() 
               if f.is_file() and f.suffix.lower() in audio_extensions)

print("Raw Data Status:\n")
print("=" * 50)

# Check each class folder
grinder_count = count_audio_files(RAW_DATA_DIR / "grinder")
background_count = count_audio_files(RAW_DATA_DIR / "background")
tools_count = count_audio_files(RAW_DATA_DIR / "tools")

print(f"Grinder recordings:     {grinder_count:>4} files")
print(f"Background sounds:      {background_count:>4} files")
print(f"Other tools:            {tools_count:>4} files")
print("=" * 50)
print(f"Total raw audio files:  {grinder_count + background_count + tools_count:>4} files\n")

if (grinder_count + background_count + tools_count) == 0:
    print("⚠️  WARNING: No audio files found!")
    print("   Please add your audio files to:")
    print(f"   - {RAW_DATA_DIR / 'grinder'}")
    print(f"   - {RAW_DATA_DIR / 'background'}")
    print(f"   - {RAW_DATA_DIR / 'tools'}")
else:
    print("✓ Raw audio files detected")

Raw Data Status:

Grinder recordings:       36 files
Background sounds:      1348 files
Other tools:              21 files
Total raw audio files:  1405 files

✓ Raw audio files detected


## 5. Load and Validate params.yaml

This configuration file contains all hyperparameters for the pipeline.

In [5]:
# Load parameters
params_file = PROJECT_ROOT / "params.yaml"

if params_file.exists():
    with open(params_file, 'r') as f:
        params = yaml.safe_load(f)
    print("✓ params.yaml loaded successfully\n")
else:
    print("⚠️  params.yaml not found. Creating default configuration...")
    params = {}

# Display key parameters
print("Configuration Overview:")
print("=" * 50)
print(json.dumps(params, indent=2))
print("=" * 50)

✓ params.yaml loaded successfully

Configuration Overview:
{
  "augmentation": {
    "grinder_multiplier": 11,
    "tools_multiplier": 9,
    "background_multiplier": 1,
    "noise_snr_min": 12,
    "noise_snr_max": 18,
    "time_stretch_min": 0.85,
    "time_stretch_max": 1.15,
    "pitch_shift_min": -3,
    "pitch_shift_max": 3,
    "gain_min": -4,
    "gain_max": 4
  },
  "preprocessing": {
    "target_sr": 16000,
    "segment_duration": 1.0,
    "segment_overlap": 0.5,
    "hop_length": 8000,
    "window_size": 16000,
    "highpass_cutoff": 80,
    "highpass_order": 5,
    "norm_method": "peak",
    "peak_level": 0.89
  },
  "features_classical": {
    "n_mfcc": 13,
    "n_fft": 512,
    "hop_length": 160,
    "spectral_rolloff_percent": 0.85,
    "spectral_contrast_bands": 6,
    "gfcc_n_filters": 100,
    "gfcc_fmin": 50,
    "gfcc_fmax": 8000,
    "gfcc_n_coeffs": 13,
    "feature_sets": [
      "mfcc",
      "gfcc",
      "combined"
    ]
  },
  "features_neural": {
    "n_fft"

## 6. Initialize DVC

DVC (Data Version Control) tracks large data files without storing them in Git.

In [6]:
# Check if DVC is already initialized
dvc_dir = PROJECT_ROOT / ".dvc"

if dvc_dir.exists():
    print("✓ DVC already initialized")
else:
    print("Initializing DVC...")
    result = subprocess.run(
        ["dvc", "init"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    if result.returncode == 0:
        print("✓ DVC initialized successfully")
    else:
        print(f"❌ DVC initialization failed:\n{result.stderr}")

✓ DVC already initialized


## 7. Configure DVC Remote Storage

**Options:**
1. **Local storage** - Store data in a separate directory (good for testing)
2. **Cloud storage** - S3, Google Cloud Storage, Azure, etc. (production)

We'll start with local storage for simplicity.

In [7]:
# Option 1: Local DVC remote (recommended for development)
local_remote_path = PROJECT_ROOT.parent / "BikeAI_dvc_storage"
local_remote_path.mkdir(exist_ok=True)

print(f"Setting up local DVC remote at: {local_remote_path}")

# Add remote
result = subprocess.run(
    ["dvc", "remote", "add", "-d", "local_storage", str(local_remote_path)],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)

if "exists" in result.stderr:
    print("✓ DVC remote already configured")
elif result.returncode == 0:
    print("✓ DVC remote configured successfully")
else:
    print(f"Note: {result.stderr}")

# Verify remote configuration
result = subprocess.run(
    ["dvc", "remote", "list"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True
)
print(f"\nConfigured DVC remotes:\n{result.stdout}")

Setting up local DVC remote at: /Users/harryirving/Development/projects/ai-ml/BikeAI_dvc_storage
✓ DVC remote already configured

Configured DVC remotes:
local_storage   /Users/harryirving/Development/dvc-storage/BikeAI/      
(default)



### Optional: Configure Cloud Storage (S3 Example)

Uncomment and configure if you want to use S3 or another cloud provider.

In [8]:
# # Option 2: AWS S3 remote (uncomment to use)
# S3_BUCKET = "s3://your-bucket-name/bikeai-data"
# 
# result = subprocess.run(
#     ["dvc", "remote", "add", "-d", "s3_storage", S3_BUCKET],
#     cwd=PROJECT_ROOT,
#     capture_output=True,
#     text=True
# )
# 
# # Configure AWS credentials (if needed)
# subprocess.run(
#     ["dvc", "remote", "modify", "s3_storage", "profile", "your-aws-profile"],
#     cwd=PROJECT_ROOT
# )

print("Cloud storage configuration skipped (using local storage)")

Cloud storage configuration skipped (using local storage)


## 8. Add Raw Data to DVC Tracking

This creates a `.dvc` file that tracks the data directory without storing it in Git.

In [9]:
# Check if data/raw is already tracked
raw_dvc_file = DATA_DIR / "raw.dvc"

if raw_dvc_file.exists():
    print("✓ data/raw already tracked by DVC")
else:
    print("Adding data/raw to DVC tracking...")
    result = subprocess.run(
        ["dvc", "add", "data/raw"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ data/raw added to DVC tracking")
        print("\nImportant: Commit the .dvc file to Git:")
        print("  git add data/raw.dvc data/.gitignore")
        print('  git commit -m "Track raw audio data with DVC"')
    else:
        print(f"❌ Failed to add data/raw:\n{result.stderr}")

✓ data/raw already tracked by DVC


## 9. Push Data to DVC Remote

Upload tracked data to the configured remote storage.

In [10]:
if (grinder_count + background_count + tools_count) > 0:
    print("Pushing data to DVC remote...")
    result = subprocess.run(
        ["dvc", "push"],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True
    )
    
    if result.returncode == 0:
        print("✓ Data pushed to DVC remote successfully")
    else:
        print(f"Note: {result.stderr}")
else:
    print("⚠️  Skipping DVC push (no data files found)")

Pushing data to DVC remote...
✓ Data pushed to DVC remote successfully


## 10. Initialize MLflow Experiment

MLflow tracks experiments, parameters, metrics, and models.

In [11]:
# Set MLflow tracking directory
MLFLOW_TRACKING_URI = f"file://{LOGS_DIR / 'mlruns'}"
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

print(f"MLflow tracking URI: {MLFLOW_TRACKING_URI}")

# Create main experiment
EXPERIMENT_NAME = "angle_grinder_pipeline"

try:
    experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
    if experiment is None:
        experiment_id = mlflow.create_experiment(
            EXPERIMENT_NAME,
            tags={
                "project": "BikeAI Theft Prevention",
                "version": "v5",
                "description": "Angle grinder detection for bike theft prevention"
            }
        )
        print(f"✓ Created MLflow experiment: {EXPERIMENT_NAME} (ID: {experiment_id})")
    else:
        print(f"✓ MLflow experiment already exists: {EXPERIMENT_NAME}")
        experiment_id = experiment.experiment_id
except Exception as e:
    print(f"❌ MLflow setup failed: {e}")
    experiment_id = None

# Set the experiment as active
mlflow.set_experiment(EXPERIMENT_NAME)

MLflow tracking URI: file:///Users/harryirving/Development/projects/ai-ml/BikeAIv5/logs/mlruns
✓ MLflow experiment already exists: angle_grinder_pipeline


<Experiment: artifact_location='file:///Users/harryirving/Development/projects/ai-ml/BikeAIv5/logs/mlruns/890722518077580718', creation_time=1765240886891, experiment_id='890722518077580718', last_update_time=1765240886891, lifecycle_stage='active', name='angle_grinder_pipeline', tags={'description': 'Angle grinder detection for bike theft prevention',
 'project': 'BikeAI Theft Prevention',
 'version': 'v5'}>

## 11. Log Setup Information to MLflow

Create a baseline run to record setup information.

In [12]:
with mlflow.start_run(run_name="00_setup_initialization") as run:
    # Log parameters from params.yaml
    if params:
        mlflow.log_params({k: v for k, v in params.items() if not isinstance(v, dict)})
    
    # Log dataset statistics
    mlflow.log_metric("raw_grinder_files", grinder_count)
    mlflow.log_metric("raw_background_files", background_count)
    mlflow.log_metric("raw_tools_files", tools_count)
    mlflow.log_metric("total_raw_files", grinder_count + background_count + tools_count)
    
    # Log setup metadata
    mlflow.set_tags({
        "stage": "setup",
        "notebook": "00_dvc_mlflow_setup",
        "timestamp": datetime.now().isoformat(),
        "dvc_initialized": str(dvc_dir.exists()),
        "data_tracked": str(raw_dvc_file.exists())
    })
    
    # Save params.yaml as artifact
    if params_file.exists():
        mlflow.log_artifact(str(params_file), artifact_path="config")
    
    print(f"✓ Setup run logged to MLflow (Run ID: {run.info.run_id})")

✓ Setup run logged to MLflow (Run ID: 8a30a028e5494d8990b6b6c14c5d8ac9)


## 12. View MLflow UI

To view your experiments and runs, open a terminal and run:

```bash
cd /Users/harryirving/Development/projects/ai-ml/BikeAIv5
mlflow ui --backend-store-uri logs/mlruns
```

Then open your browser to: http://localhost:5000

## 13. Setup Summary and Next Steps

In [13]:
print("\n" + "=" * 60)
print("SETUP SUMMARY")
print("=" * 60)

# Check each component
checks = [
    ("Directory structure", all(d.exists() for d in required_dirs)),
    ("DVC initialized", dvc_dir.exists()),
    ("DVC remote configured", True),  # Already verified above
    ("Raw data tracked", raw_dvc_file.exists() or (grinder_count + background_count + tools_count) == 0),
    ("MLflow experiment created", experiment_id is not None),
    ("params.yaml loaded", params_file.exists()),
]

for check_name, status in checks:
    icon = "✓" if status else "❌"
    print(f"{icon} {check_name}")

print("\n" + "=" * 60)
print("NEXT STEPS")
print("=" * 60)
print("\n1. Add your raw audio files (if not done):")
print(f"   - Grinder recordings → {RAW_DATA_DIR / 'grinder'}")
print(f"   - Background sounds → {RAW_DATA_DIR / 'background'}")
print(f"   - Other tools → {RAW_DATA_DIR / 'tools'}")
print("\n2. Run DVC commands (if you added new data):")
print("   dvc add data/raw")
print("   git add data/raw.dvc data/.gitignore")
print('   git commit -m "Track raw audio data"')
print("   dvc push")
print("\n3. Proceed to next notebook:")
print("   → 01_data_exploration.ipynb")
print("\n4. View MLflow UI:")
print("   mlflow ui --backend-store-uri logs/mlruns")
print("   Then visit: http://localhost:5000")
print("\n" + "=" * 60)

print("\n✓ Setup complete! Ready to begin data exploration.")


SETUP SUMMARY
✓ Directory structure
✓ DVC initialized
✓ DVC remote configured
✓ Raw data tracked
✓ MLflow experiment created
✓ params.yaml loaded

NEXT STEPS

1. Add your raw audio files (if not done):
   - Grinder recordings → /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/raw/grinder
   - Background sounds → /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/raw/background
   - Other tools → /Users/harryirving/Development/projects/ai-ml/BikeAIv5/data/raw/tools

2. Run DVC commands (if you added new data):
   dvc add data/raw
   git add data/raw.dvc data/.gitignore
   git commit -m "Track raw audio data"
   dvc push

3. Proceed to next notebook:
   → 01_data_exploration.ipynb

4. View MLflow UI:
   mlflow ui --backend-store-uri logs/mlruns
   Then visit: http://localhost:5000


✓ Setup complete! Ready to begin data exploration.


---

## Additional Notes

### DVC Workflow Summary

**When you add/modify **
```bash
dvc add data/raw              # Track changes
git add data/raw.dvc          # Stage .dvc file
git commit -m "Update data"   # Commit metadata
dvc push                      # Upload to remote
```

**When collaborating:**
```bash
git pull                      # Get .dvc file updates
dvc pull                      # Download actual data
```

### MLflow Experiment Naming Convention

Throughout this project, we'll use consistent experiment names:
- `angle_grinder_pipeline` - Main experiment
- `classical_<feat_set>_<model>` - Classical ML experiments
- `neural_<architecture>` - Neural network experiments

### Parameters Management

All hyperparameters are centralized in `params.yaml`. When you modify parameters:
1. Edit `params.yaml`
2. Commit to Git
3. MLflow will automatically log the new parameters in each run

This enables easy experiment comparison and reproducibility.

---

**Notebook Status:** ✓ Complete

In [14]:
import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())


Tracking URI: file:///Users/harryirving/Development/projects/ai-ml/BikeAIv5/logs/mlruns
